In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import pygeohash as pgh
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)

data_paths = [".", "data/raw", "../../data/raw"]
base_path = next((path for path in data_paths if os.path.exists(os.path.join(path, "train.csv"))), None)

raw_train = pd.read_csv(os.path.join(base_path, "train.csv"))
raw_test = pd.read_csv(os.path.join(base_path, "test.csv"))

y_train = raw_train['demand'].values
submission_index = raw_test['Index'].values

print("Core datasets loaded. Target baseline extracted.")

Core datasets loaded. Target baseline extracted.


In [2]:
def engineer_features(df):
    df_feat = df.copy()
    
    # 1. Cyclical Time Kinematics
    time_split = df_feat['timestamp'].str.split(':', expand=True).astype(int)
    df_feat['ts_minutes'] = time_split[0] * 60 + time_split[1]
    df_feat['hour'] = time_split[0]
    df_feat['time_slot_15m'] = df_feat['ts_minutes'] // 15
    
    df_feat['hour_sin'] = np.sin(2 * np.pi * df_feat['hour'] / 24.0)
    df_feat['hour_cos'] = np.cos(2 * np.pi * df_feat['hour'] / 24.0)
    df_feat['min_sin'] = np.sin(2 * np.pi * df_feat['ts_minutes'] / 1440.0)
    df_feat['min_cos'] = np.cos(2 * np.pi * df_feat['ts_minutes'] / 1440.0)
    
    # 2. Continuous Spatial Signal
    coords = df_feat['geohash'].apply(lambda x: pgh.decode(x) if isinstance(x, str) else (np.nan, np.nan))
    df_feat['lat'] = coords.apply(lambda x: x[0])
    df_feat['lon'] = coords.apply(lambda x: x[1])
    
    # 3. The Target Interaction Key
    df_feat['geo_time_interaction'] = df_feat['geohash'].astype(str) + "_" + df_feat['time_slot_15m'].astype(str)
    
    # 4. Standard Fallbacks
    df_feat['Temperature'] = df_feat['Temperature'].fillna(df_feat['Temperature'].median())
    for col in ['Weather', 'RoadType', 'LargeVehicles', 'Landmarks']:
        df_feat[col] = df_feat[col].fillna('Unknown')
        
    return df_feat

X_train_fe = engineer_features(raw_train.drop(columns=['demand'], errors='ignore'))
X_test_fe = engineer_features(raw_test)

print("Spatiotemporal kinematics and interaction keys synthesized.")

Spatiotemporal kinematics and interaction keys synthesized.


In [3]:
def get_oof_target_encoding(train_df, test_df, target, column_name, folds=5):
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    
    train_encoded = np.zeros(len(train_df))
    test_encoded = np.zeros(len(test_df))
    
    temp_train = train_df[[column_name]].copy()
    temp_test = test_df[[column_name]].copy()
    temp_train['target'] = target
    
    global_mean = target.mean()
    
    # Map the entire training mean to the test set
    test_mapping = temp_train.groupby(column_name)['target'].mean()
    test_encoded = temp_test[column_name].map(test_mapping).fillna(global_mean).values
    
    # K-Fold mapping for training set to prevent self-leakage
    for train_idx, val_idx in kf.split(temp_train):
        fold_train = temp_train.iloc[train_idx]
        fold_val = temp_train.iloc[val_idx]
        
        fold_mapping = fold_train.groupby(column_name)['target'].mean()
        train_encoded[val_idx] = fold_val[column_name].map(fold_mapping).fillna(global_mean).values
        
    return train_encoded, test_encoded

# Apply OOF Encoding to the Golden Feature
X_train_fe['TE_geo_time'], X_test_fe['TE_geo_time'] = get_oof_target_encoding(
    X_train_fe, X_test_fe, y_train, 'geo_time_interaction'
)

# Apply OOF Encoding to base Geohash
X_train_fe['TE_geohash'], X_test_fe['TE_geohash'] = get_oof_target_encoding(
    X_train_fe, X_test_fe, y_train, 'geohash'
)

# Categorical Label Encoding
cat_cols = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for c in cat_cols:
    le = LabelEncoder()
    combined = X_train_fe[c].astype(str).tolist() + X_test_fe[c].astype(str).tolist()
    le.fit(combined)
    X_train_fe[c] = le.transform(X_train_fe[c].astype(str))
    X_test_fe[c] = le.transform(X_test_fe[c].astype(str))

# Matrix Formatting
drop_cols = ['timestamp', 'geo_time_interaction', 'Index']
features = [c for c in X_train_fe.columns if c not in drop_cols]

X = X_train_fe[features].values
X_test = X_test_fe[features].values

print(f"Matrix locked and leak-free. Shape: {X.shape}")

Matrix locked and leak-free. Shape: (77299, 19)


In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Reverting to the proven RMSE objectives with high regularization
lgb_p = {'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.03, 'max_depth': 8, 'num_leaves': 128, 'min_child_samples': 20, 'verbose': -1, 'random_state': 42, 'n_jobs': -1}
xgb_p = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.03, 'max_depth': 7, 'random_state': 42, 'n_jobs': -1}
cat_p = {'iterations': 2500, 'learning_rate': 0.03, 'depth': 8, 'eval_metric': 'RMSE', 'verbose': 0, 'random_seed': 42}

oof_lgb, test_lgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_xgb, test_xgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_cat, test_cat = np.zeros(len(X)), np.zeros(len(X_test))

print("Executing 5-Fold Validation...")

for fold, (t_idx, v_idx) in enumerate(kf.split(X)):
    X_tr, y_tr = X[t_idx], y_train[t_idx]
    X_va, y_va = X[v_idx], y_train[v_idx]
    
    m_lgb = lgb.train(lgb_p, lgb.Dataset(X_tr, y_tr), 2500, valid_sets=[lgb.Dataset(X_va, y_va)], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[v_idx] = m_lgb.predict(X_va)
    test_lgb += m_lgb.predict(X_test) / 5
    
    m_xgb = xgb.train(xgb_p, xgb.DMatrix(X_tr, y_tr), 2500, evals=[(xgb.DMatrix(X_va, y_va), 'val')], early_stopping_rounds=100, verbose_eval=False)
    oof_xgb[v_idx] = m_xgb.predict(xgb.DMatrix(X_va))
    test_xgb += m_xgb.predict(xgb.DMatrix(X_test)) / 5
    
    m_cat = CatBoostRegressor(**cat_p).fit(X_tr, y_tr, eval_set=(X_va, y_va))
    oof_cat[v_idx] = m_cat.predict(X_va)
    test_cat += m_cat.predict(X_test) / 5
    
    fold_blend = (oof_lgb[v_idx] + oof_xgb[v_idx] + oof_cat[v_idx]) / 3.0
    print(f"Fold {fold+1} R2 Score: {max(0, 100 * r2_score(y_va, fold_blend)):.4f}")

Executing 5-Fold Validation...
Fold 1 R2 Score: 95.6562
Fold 2 R2 Score: 95.5512
Fold 3 R2 Score: 95.6259
Fold 4 R2 Score: 95.2585
Fold 5 R2 Score: 95.4826


In [6]:
def obj_func(w):
    w = np.array(w)
    if w.sum() == 0: return 999.0
    w_norm = w / w.sum()
    blend = (w_norm[0] * oof_lgb) + (w_norm[1] * oof_xgb) + (w_norm[2] * oof_cat)
    return -max(0, 100 * r2_score(y_train, blend))

w_opt = minimize(obj_func, [0.33, 0.33, 0.33], method='Nelder-Mead').x
w_opt /= sum(w_opt)

final_preds = np.clip((w_opt[0]*test_lgb + w_opt[1]*test_xgb + w_opt[2]*test_cat), 0.0, 1.0)
final_r2 = max(0, 100 * r2_score(y_train, (w_opt[0]*oof_lgb + w_opt[1]*oof_xgb + w_opt[2]*oof_cat)))

pd.DataFrame({'Index': submission_index, 'demand': final_preds}).to_csv("submission_v10.csv", index=False)

print("\nPipeline execution complete.")
print(f"Optimized Terminal R2 Score: {final_r2:.4f}")
print("Output generated: submission_v10.csv")


Pipeline execution complete.
Optimized Terminal R2 Score: 95.5310
Output generated: submission_v10.csv
